# Actor Role Chi-square: Non-independence Robustness (R2-3.1)

Tests the association between account type and structural role (chi2 = 275.41, df = 4) against network-aware nulls:
1. global node-label permutation of account type (10,000 permutations, seed 42);
2. permutation of account type within Louvain communities, preserving community composition;
3. community cluster bootstrap 95% CI for Cramer's V (B = 10,000).

Input: `derived-data/hashed/actor_roles_classification_hashed.csv`.

In [1]:
import pandas as pd, numpy as np, time
from scipy.stats import chi2_contingency
df = pd.read_csv('actor_roles_classification_hashed.csv')  # derived-data/hashed/
df['structural_role'] = df['structural_role'].str.strip()
t = pd.Categorical(df['account_type']); r = pd.Categorical(df['structural_role'])
ti, ri = t.codes, r.codes; nt, nr = len(t.categories), len(r.categories)
comm = df['community'].values; N = len(df)
def tab(a, b): return np.bincount(a * nr + b, minlength=nt * nr).reshape(nt, nr)
def chi(tb):
    e = tb.sum(1, keepdims=True) * tb.sum(0, keepdims=True) / tb.sum()
    return ((tb - e) ** 2 / e).sum()
def V(tb): return np.sqrt(chi(tb) / (tb.sum() * (min(tb.shape) - 1)))
obs = tab(ti, ri)
chi2, p, dof, exp = chi2_contingency(obs)
print('N', N, '| types', list(t.categories), '| roles', list(r.categories))
print(pd.DataFrame(obs, index=t.categories, columns=r.categories))
print(f'chi2={chi2:.2f} df={dof} p={p:.3g} V={V(obs):.3f} | min expected={exp.min():.1f}')
B = 10000; rng = np.random.default_rng(42)


N 11426 | types ['elite/political', 'grassroot', 'media/news'] | roles ['Attention Sink', 'Diffusion Driver', 'Local Amplifier']
                 Attention Sink  Diffusion Driver  Local Amplifier
elite/political             102                 6                9
grassroot                  4043              1002             6144
media/news                  108                 2               10
chi2=275.41 df=4 p=2.17e-58 V=0.110 | min expected=10.3


In [2]:
# (1) node-label permutation: shuffle account_type across all nodes
t0 = time.time(); null = np.array([chi(tab(rng.permutation(ti), ri)) for _ in range(B)])
print(f'[1] global label permutation: max null chi2={null.max():.2f}, mean={null.mean():.2f}, null>=obs: {(null >= chi2).sum()}/{B}, p_perm={(np.sum(null >= chi2) + 1) / (B + 1):.2e} ({time.time()-t0:.0f}s)')


[1] global label permutation: max null chi2=29.30, mean=3.98, null>=obs: 0/10000, p_perm=1.00e-04 (5s)


In [3]:
# (2) stratified permutation within communities (preserves community composition)
order = np.argsort(comm, kind='stable'); cs = comm[order]
bounds = np.flatnonzero(np.r_[True, cs[1:] != cs[:-1], True])
t0 = time.time(); null2 = np.empty(B); ti_sorted = ti[order]; ri_sorted = ri[order]
for k in range(B):
    perm = ti_sorted.copy()
    for a, b in zip(bounds[:-1], bounds[1:]):
        if b - a > 1: perm[a:b] = rng.permutation(perm[a:b])
    null2[k] = chi(tab(perm, ri_sorted))
print(f'[2] within-community permutation: max null chi2={null2.max():.2f}, mean={null2.mean():.2f}, null>=obs: {(null2 >= chi2).sum()}/{B}, p_perm={(np.sum(null2 >= chi2) + 1) / (B + 1):.2e} ({time.time()-t0:.0f}s)')


[2] within-community permutation: max null chi2=57.00, mean=11.47, null>=obs: 0/10000, p_perm=1.00e-04 (103s)


In [4]:
# (3) community cluster bootstrap CI for Cramer's V
uc, inv = np.unique(comm, return_inverse=True); groups = [np.flatnonzero(inv == g) for g in range(len(uc))]
t0 = time.time(); vs = np.empty(B)
for k in range(B):
    pick = rng.integers(0, len(uc), len(uc)); idx = np.concatenate([groups[g] for g in pick])
    vs[k] = V(tab(ti[idx], ri[idx]))
lo, hi = np.percentile(vs, [2.5, 97.5])
print(f'[3] community cluster bootstrap V 95% CI [{lo:.3f}, {hi:.3f}] (B={B}, {len(uc)} clusters, {time.time()-t0:.0f}s)')


[3] community cluster bootstrap V 95% CI [0.082, 0.136] (B=10000, 1667 clusters, 10s)
